In [39]:
import numpy as np
import pandas as pd
import requests
import plotly.graph_objects as go
from astropy.time import Time
from astropy.coordinates import TEME, ITRS, EarthLocation, CartesianRepresentation
import astropy.units as u
from datetime import datetime, timezone
from sgp4.api import Satrec



In [6]:
ST_BASE    = "https://www.space-track.org"
ST_LOGIN   = f"{ST_BASE}/ajaxauth/login"
ST_QUERY   = f"{ST_BASE}/basicspacedata/query"

USERNAME = "nathandarby2022@gmail.com"   # replace with your Space-Track credentials
PASSWORD = "Boobear32*Boobear32"             # consider loading from env variable

session = requests.Session()

resp = session.post(ST_LOGIN, data={"identity": "nathandarby2022@gmail.com", "password": "Boobear32*Boobear32"})

if resp.status_code == 200:
    print("Logged in to Space-Track successfully!")
else:
    print(f"Login failed: {resp.status_code}")

Logged in to Space-Track successfully!


In [7]:
resp = session.post(ST_LOGIN, data={"identity": "nathandarby2022@gmail.com", "password": "Boobear32*Boobear32"})
print(resp.status_code)

200


In [37]:
import json
import os
from datetime import datetime, timezone

SATCAT_CACHE_FILE = os.path.expanduser("~/satcat_cache.json")

def get_norad_ids(intldes, session, force=False):
    """
    Get all NORAD IDs for a launch group.
    Reads from local cache if available, otherwise queries Space-Track satcat.
    """
    # Check cache first
    if os.path.exists(SATCAT_CACHE_FILE) and not force:
        with open(SATCAT_CACHE_FILE) as f:
            cache = json.load(f)
        
        if cache.get('intldes') == intldes:
            print(f"Loaded {len(cache['norad_ids'])} NORAD IDs from cache")
            print(f"   Launch:    {intldes}")
            print(f"   Cached at: {cache['fetched_at']}")
            for obj in cache['objects']:
                print(f"   {obj['NORAD_CAT_ID']} — {obj['SATNAME']}")
            return cache['norad_ids']
        else:
            print(f"Cache is for different launch ({cache.get('intldes')}) — fetching fresh")

    # Query satcat
    print(f"Querying Space-Track satcat for launch {intldes}...")
    url = (f"{ST_QUERY}/class/satcat"
           f"/INTLDES/{intldes}~~"
           f"/OBJECT_TYPE/Payload"
           f"/orderby/NORAD_CAT_ID"
           f"/format/json")
    
    resp = session.get(url)
    
    if resp.status_code != 200:
        print(f"satcat query failed: {resp.status_code}")
        return []
    
    objects = resp.json()
    norad_ids = [obj['NORAD_CAT_ID'] for obj in objects]
    
    # Save to cache
    cache = {
        'intldes':    intldes,
        'fetched_at': datetime.now(timezone.utc).isoformat(),
        'norad_ids':  norad_ids,
        'objects':    [{'NORAD_CAT_ID': obj['NORAD_CAT_ID'], 
                        'SATNAME':      obj['SATNAME']} for obj in objects]
    }
    with open(SATCAT_CACHE_FILE, 'w') as f:
        json.dump(cache, f, indent=2)
    
    print(f"Fetched and cached {len(norad_ids)} payloads for launch {intldes}")
    for obj in objects:
        print(f"   {obj['NORAD_CAT_ID']} — {obj['SATNAME']}")
    
    return norad_ids

In [38]:
TLE_CACHE_FILE = os.path.expanduser("~/tle_cache.json")

def fetch_tles(norad_ids, session, chunk_size=50, force=False):
    if os.path.exists(TLE_CACHE_FILE) and not force:
        with open(TLE_CACHE_FILE) as f:
            cache = json.load(f)
        fetched_at = datetime.fromisoformat(cache['fetched_at'])
        age = datetime.now(timezone.utc) - fetched_at
        if age.total_seconds() < 3600:
            remaining = 3600 - age.total_seconds()
            print(f"Loaded TLEs from Cache")
            print(f"   Last Query:            {int(age.total_seconds()//60)} min ago")
            print(f"   Next Available Query:  {int(remaining//60)} min")
            print(f"   Total TLEs:            {len(cache['tles'])}")
            return cache['tles']
        else:
            print("Cache expired — fetching fresh TLEs")

    all_tles = []
    for i in range(0, len(norad_ids), chunk_size):
        chunk = norad_ids[i:i+chunk_size]
        ids_str = ','.join(str(n) for n in chunk)
        url = (f"{ST_QUERY}/class/gp"
               f"/NORAD_CAT_ID/{ids_str}"
               f"/decay_date/null-val"
               f"/orderby/NORAD_CAT_ID"
               f"/format/json")
        resp = session.get(url)
        print(f"   Chunk {i//chunk_size + 1}: status {resp.status_code}")
        if resp.status_code != 200 or not resp.text.strip():
            continue
        for obj in resp.json():
            if obj.get('TLE_LINE1') and obj.get('TLE_LINE2'):
                all_tles.append({
                    'name':  obj['OBJECT_NAME'],
                    'line1': obj['TLE_LINE1'],
                    'line2': obj['TLE_LINE2']
                })

    # Save to cache
    cache = {'fetched_at': datetime.now(timezone.utc).isoformat(), 'tles': all_tles}
    with open(TLE_CACHE_FILE, 'w') as f:
        json.dump(cache, f, indent=2)

    print(f"\nLoaded TLEs from Query")
    print(f"   Last Query:            just now")
    print(f"   Next Available Query:  60 min")
    print(f"   Total TLEs:            {len(all_tles)}")
    return all_tles

In [34]:
# Transporter-9 — Oct 2023
norad_ids = get_norad_ids('2023-174', session, force=False)
tles = fetch_tles(norad_ids, session, force=False)

Loaded 95 NORAD IDs from cache
   Launch:    2023-174
   Cached at: 2026-04-21T03:54:00.131847+00:00
   58256 — CONNECTA T3.1
   58257 — MANTIS
   58258 — ORBASTRO-PC-1
   58259 — LEMUR 2 BASS
   58260 — LEMUR 2 DILIGHTFUL
   58262 — PLATERO
   58263 — LEMUR 2 GOOD-VIBES
   58264 — BRO-11
   58266 — DJIBOUTI-1A
   58267 — LEMUR 2 SANITA-VETRA
   58268 — CONNECTA T3.2
   58269 — IRIS-C2
   58270 — FLOCK 4Q 1
   58271 — FLOCK 4Q 16
   58272 — TIGER-6
   58273 — FLOCK 4Q 11
   58274 — FLOCK 4Q 36
   58275 — FLOCK 4Q 12
   58276 — GENMAT-1
   58277 — TIGER-5
   58278 — FLOCK 4Q 19
   58279 — FLOCK 4Q 8
   58280 — FLOCK 4Q 28
   58281 — AETHER-1
   58282 — FLOCK 4Q 27
   58283 — FLOCK 4Q 17
   58284 — FLOCK 4Q 26
   58285 — FLOCK 4Q 31
   58286 — FLOCK 4Q 30
   58287 — PICO-01B009
   58288 — ICEYE-X31
   58291 — FALCONSAT-10
   58292 — UMBRA-08
   58293 — ICEYE-X32
   58294 — ICEYE-X34
   58295 — SPACEVAN-001
   58296 — PELICAN-1 3001
   58297 — UMBRA-07
   58298 — SPIP
   58299 — AETHER-2


In [40]:
rows = []
for t in tles:
    line1 = t['line1']
    intldes_raw = line1[9:17].strip()
    year  = intldes_raw[:2]
    num   = intldes_raw[2:5]
    piece = intldes_raw[5:]
    full_year = f"19{year}" if int(year) >= 57 else f"20{year}"
    intldes = f"{full_year}-{num}{piece}"
    
    rows.append({
        'International Designator': intldes,
        'NORAD Catalog Number':     line1[2:7].strip(),
        'Name':                     t['name'],
        'TLE Line 1':               line1,
        'TLE Line 2':               t['line2']
    })

df_tles = pd.DataFrame(rows)
print(f"Total objects: {len(df_tles)}")
df_tles

Total objects: 55


,International Designator,NORAD Catalog Number,Name,TLE Line 1,TLE Line 2
0,2023-174C,58258,ORBASTRO-PC-1,1 58258U 23174C 26110.96595891 .00021955 0...,2 58258 97.3773 195.8869 0006387 29.8045 330...
1,2023-174D,58259,LEMUR 2 BASS,1 58259U 23174D 26110.46033937 .00010571 0...,2 58259 97.3841 194.4526 0007066 30.3338 329...
2,2023-174G,58262,PLATERO,1 58262U 23174G 26110.87393257 .00027980 0...,2 58262 97.3917 206.4394 0003482 319.5644 40...
3,2023-174J,58264,BRO-11,1 58264U 23174J 26110.91099356 .00004932 0...,2 58264 97.3913 188.3020 0011648 56.6536 303...
4,2023-174Q,58270,FLOCK 4Q 1,1 58270U 23174Q 26110.66425710 .00086422 0...,2 58270 97.3811 202.0003 0003881 352.2299 7...
5,2023-174R,58271,FLOCK 4Q 16,1 58271U 23174R 26110.18342994 .00014614 0...,2 58271 97.3823 197.0630 0005814 26.4692 333...
6,2023-174S,58272,TIGER-6,1 58272U 23174S 26110.80384366 .00194258 0...,2 58272 97.3863 206.6630 0003290 267.8435 92...
7,2023-174T,58273,FLOCK 4Q 11,1 58273U 23174T 26110.50295132 .00017697 0...,2 58273 97.3874 197.9207 0005438 23.6552 336...
8,2023-174U,58274,FLOCK 4Q 36,1 58274U 23174U 26110.50416786 .00028399 0...,2 58274 97.3847 200.9724 0005195 355.7476 4...
9,2023-174V,58275,FLOCK 4Q 12,1 58275U 23174V 26110.80547331 .00091522 0...,2 58275 97.3832 201.3914 0002588 317.4886 42...
